In [4]:
# Cell 1 — inspect the actual saved filenames
import os
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

XAI_DIR = os.path.abspath('.')
if not os.path.isdir(os.path.join(XAI_DIR, 'results')):
    XAI_DIR = os.path.abspath('XAI')
RESULTS_DIR = os.path.join(XAI_DIR, 'results')
CSV_PATH = os.path.join(RESULTS_DIR, 'xai_manifest.csv')

for group in ('real', 'fake'):
    group_dir = os.path.join(RESULTS_DIR, group)
    samples = sorted(name for name in os.listdir(group_dir) if os.path.isdir(os.path.join(group_dir, name))) if os.path.isdir(group_dir) else []
    if samples:
        sample_folder = os.path.join(group_dir, samples[0])
        print(f'Actual filenames in {sample_folder}:')
        print(sorted(os.listdir(sample_folder)))
        break


Actual filenames in d:\PROJECTS\DeepFake-Detector-main\XAI\results\real\image1:
['contact_sheet.png', 'gradcam_heatmap.png', 'gradcam_overlay.png', 'gradcam_plus_plus_heatmap.png', 'gradcam_plus_plus_overlay.png', 'input.png', 'integrated_gradients_heatmap.png', 'integrated_gradients_overlay.png', 'lime_heatmap.png', 'lime_overlay.png', 'score_cam_heatmap.png', 'score_cam_overlay.png']


In [5]:
# Cell 2 — robust discovery and image loading helpers
IMAGE_SIZE = (256, 256)
METHODS = [('original', 'Original'), ('gradcam', 'Grad-CAM'), ('gradcam_plus_plus', 'Grad-CAM++'), ('score_cam', 'Score-CAM'), ('integrated_gradients', 'Integrated Gradients'), ('lime', 'LIME')]
FILENAME_CANDIDATES = {
    'original': ['original.png', 'input.png', 'image.png', 'source.png'],
    'gradcam': ['gradcam.png', 'grad_cam.png', 'gradcam_overlay.png'],
    'gradcam_plus_plus': ['gradcam_plus_plus.png', 'gradcam++.png', 'gradcampp.png', 'grad_cam_plus_plus.png', 'gradcam_plus_plus_overlay.png'],
    'score_cam': ['score_cam.png', 'scorecam.png', 'score_cam_overlay.png'],
    'integrated_gradients': ['integrated_gradients.png', 'integratedgradients.png', 'ig.png', 'integrated_gradients_overlay.png'],
    'lime': ['lime.png', 'lime_overlay.png'],
}

def normalise_name(filename):
    return ''.join(ch for ch in os.path.splitext(filename.lower())[0] if ch.isalnum())

def find_image_file(folder, method):
    if not os.path.isdir(folder):
        return None
    for candidate in FILENAME_CANDIDATES[method]:
        path = os.path.join(folder, candidate)
        if os.path.isfile(path):
            return path
    keys = {normalise_name(name) for name in FILENAME_CANDIDATES[method]}
    for name in os.listdir(folder):
        if name.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')) and normalise_name(name) in keys:
            return os.path.join(folder, name)
    return None

def show_image_or_missing(ax, image_path):
    ax.set_xticks([]); ax.set_yticks([])
    if image_path:
        try:
            with Image.open(image_path) as image:
                ax.imshow(image.convert('RGB').resize(IMAGE_SIZE, Image.Resampling.BILINEAR))
            return
        except Exception:
            pass
    ax.imshow(np.full((256, 256, 3), 235, dtype=np.uint8))
    ax.text(128, 128, 'missing', ha='center', va='center', fontsize=12, bbox={'facecolor': 'white', 'edgecolor': 'gray'})

def sample_sort_key(name):
    digits = ''.join(ch for ch in name if ch.isdigit())
    return (int(digits) if digits else 9999, name)

def discover_samples():
    samples = []
    for group in ('real', 'fake'):
        group_dir = os.path.join(RESULTS_DIR, group)
        names = sorted([name for name in os.listdir(group_dir) if os.path.isdir(os.path.join(group_dir, name))], key=sample_sort_key) if os.path.isdir(group_dir) else []
        samples.extend({'group': group.upper(), 'image_name': name, 'folder': os.path.join(group_dir, name)} for name in names)
    return samples


In [6]:
# Cell 3 — 10-row × 6-column comparison grid
samples = discover_samples()
fig, axes = plt.subplots(len(samples), len(METHODS), figsize=(16, max(20, len(samples) * 2.65)))
axes = np.atleast_2d(axes)
for row_index, sample in enumerate(samples):
    for col_index, (method_key, method_title) in enumerate(METHODS):
        ax = axes[row_index, col_index]
        show_image_or_missing(ax, find_image_file(sample['folder'], method_key))
        if row_index == 0:
            ax.set_title(method_title, fontsize=13, fontweight='bold', pad=10)
        if col_index == 0:
            ax.set_ylabel(f"{sample['group']} {sample['image_name']}", fontsize=11, fontweight='bold', rotation=0, labelpad=70, va='center')
fig.suptitle('Wide Model XAI Comparison', fontsize=18, fontweight='bold', y=0.995)
fig.tight_layout(rect=[0.06, 0, 1, 0.99])
grid_output = os.path.join(XAI_DIR, 'xai_comparison_grid.png')
fig.savefig(grid_output, dpi=180, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {grid_output}')


Saved: d:\PROJECTS\DeepFake-Detector-main\XAI\xai_comparison_grid.png


In [7]:
# Cell 4 — failure-case figure: fake image predicted REAL

df = pd.read_csv(CSV_PATH)

failure_rows = df[
    (df["ground_truth_folder"].astype(str).str.upper() == "FAKE") &
    (df["wide_prediction"].astype(str).str.upper() == "REAL")
]

if not failure_rows.empty:
    failure = failure_rows.iloc[0]
    failure_folder = os.path.join(
        XAI_DIR,
        failure["evidence_folder"]
    ) if not os.path.isabs(str(failure["evidence_folder"])) else str(failure["evidence_folder"])
else:
    # Fallback requested by the task.
    failure_folder = os.path.join(RESULTS_DIR, "fake", "image1")
    print("No FAKE→REAL row found in CSV; using fake/image1 fallback.")

fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))

for ax, (method_key, method_title) in zip(axes, METHODS):
    image_path = find_image_file(failure_folder, method_key)
    show_image_or_missing(ax, image_path)
    ax.set_title(method_title, fontsize=10, fontweight="bold")

fig.suptitle(
    "Failure Case: Fake Misclassified as Real",
    fontsize=16,
    fontweight="bold",
    y=1.04
)
fig.tight_layout()

failure_output = os.path.join(XAI_DIR, "xai_failure_case.png")
fig.savefig(failure_output, dpi=180, bbox_inches="tight")
plt.close(fig)

print(f"Saved: {failure_output}")

Saved: d:\PROJECTS\DeepFake-Detector-main\XAI\xai_failure_case.png


In [8]:
# Cell 5 — attribution summary table and sample accuracy

method_columns = [
    "gradcam",
    "gradcam_plus_plus",
    "score_cam",
    "integrated_gradients",
    "lime",
]

summary_rows = []

for group_name, group_df in df.groupby("ground_truth_folder"):
    row = {"Ground truth": str(group_name).upper()}

    for method in method_columns:
        column = f"{method}_mean"

        if column in group_df.columns:
            values = pd.to_numeric(group_df[column], errors="coerce").dropna()
            row[method] = (
                f"{values.mean():.4f} ± {values.std(ddof=1):.4f}"
                if len(values) > 1
                else f"{values.mean():.4f} ± 0.0000"
            )
        else:
            row[method] = "missing column"

    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows).set_index("Ground truth")
summary_table.columns = [
    "Grad-CAM",
    "Grad-CAM++",
    "Score-CAM",
    "Integrated Gradients",
    "LIME",
]

print("\nXAI mean-attribution summary (mean ± std):")
print(summary_table.to_string())

correct_count = (
    df["correct"].astype(str).str.strip().str.lower().eq("yes").sum()
)
print(f"\nXAI sample accuracy: {correct_count}/{len(df)}")


XAI mean-attribution summary (mean ± std):
                     Grad-CAM       Grad-CAM++        Score-CAM Integrated Gradients             LIME
Ground truth                                                                                         
FAKE          0.1058 ± 0.0144  0.0751 ± 0.0207  0.2453 ± 0.0750      0.1271 ± 0.0256  0.3094 ± 0.1641
REAL          0.1275 ± 0.0546  0.0860 ± 0.0402  0.3294 ± 0.1652      0.1230 ± 0.0235  0.1075 ± 0.0789

XAI sample accuracy: 9/10
